### Introducción al problema

Seguramente has estado en esta situación: tienes que recopilar el estatus histórico o acumulado del rubro que estás revisando, adicional a que debes tomar en cuenta las altas, bajas y modificaciones del periodo bajo revisión, esto lo debes multiplicar por el número de almacenes, unidades de negocio, países, o lo que sea que estés revisando, tú nómbralo. Al final todo lo debes condensar en una tabla final que ira directo a una presentación ejecutiva mensual para la alta dirección. 

En mi caso esta tarea era habitual, mes a mes, pero se trataba de un inventario regulatorio (normatividad de leyes aplicables) en una institución bancaria, cuya granularidad era alta puesto que se trataba de artículos especifico de legislación bancaria clave, normalmente 50 artículos por 15 unidades de negocio, algunos coincidian entre unidades, pero otros eran muy específicos a Mercados, otros muy específicos con Comercio, área Fiduciaria, etc. **La responsabilidad era crítica**.

Voy a tratar de sobre-simplificar las cosas: sea lo que revises tienes varios estatus y debes ser consistente con la fecha de corte anterior, de ahí tomar estos estatus como un inicio y debes indicar si hay altas, bajas y "en proceso". Si lo haces mes a mes, pues la mejor opción es automatizarlo.

In [1]:
import pandas as pd
import numpy as np

In [ ]:
# importamos los datos del periodo anterior
anterior = pd.read_excel(r'data\mtto_inv_reg.xlsx', sheet_name='anterior')
anterior.head()

,Publication ID,Publication Name,Regulatory Tier,Obligation ID,Control Mapping Status
0,3345,Ley Renta,Tier 1,2037.0,Confirmado
1,3345,Ley Renta,Tier 1,2488.0,Pending
2,3467,Ley IVA,Tier 1,2700.0,Confirmado
3,3467,Ley IVA,Tier 1,2105.0,Confirmado
4,3467,Ley IVA,Tier 1,2315.0,Confirmado


In [ ]:
# importamos los datos del periodo actual
actual = pd.read_excel(r'data\mtto_inv_reg.xlsx', sheet_name='actual')
actual.head()

,Publication ID,Publication Name,Regulatory Tier,Obligation ID,Control Mapping Status
0,3345,Ley Renta,Tier 1,2037.0,Confirmado
1,3345,Ley Renta,Tier 1,2488.0,Confirmado
2,3467,Ley IVA,Tier 1,2700.0,Confirmado
3,3467,Ley IVA,Tier 1,2315.0,Confirmado
4,3467,Ley IVA,Tier 1,2990.0,Pending


donde:
- Publication ID -> es un identificador de la Ley, circular, reglamento, etc., es el documento que contiene artículos
- Publication Name -> nombre de la Ley, circular, reglamento, etc.
- Regulatory Tier -> importancia de la Ley, circular, reglamento, etc., Tier 1 es el más importante. Tier 3 no contiene artículos.
- Obligation ID -> es un identificador a un artículo contenido en una Ley, circular, reglamento, etc.
- Control Mapping Status -> se trata del estatus actual en el inventario regulatorio: Confirmado ó Pending

In [ ]:
# creación de la función

def estatus_inventario(anterior, actual):
    """
    Compara dos inventarios y devuelve métricas por publicación,
    con notas sobre altas, bajas y obligaciones pendientes.

    Requiere:
        Publication ID, Publication Name, Regulatory Tier,
        Obligation ID y Control Mapping Status.
    """

    # 1. preparar copias y normalizar columnas
    renombres = {
        "Publication Name": "Nombre",
        "Regulatory Tier": "Tier",
        "Control Mapping Status": "Status",
    }

    requeridas = {
        "Publication ID", "Nombre", "Tier",
        "Obligation ID", "Status"
    }

    def preparar(df):
        df = df.copy()
        df.columns = df.columns.str.strip()
        df = df.rename(columns=renombres)

        if df.columns.duplicated().any():
            raise ValueError("Hay nombres de columnas duplicados.")

        faltantes = requeridas - set(df.columns)
        if faltantes:
            raise ValueError(
                f"Faltan columnas requeridas: {sorted(faltantes)}"
            )

        df["Obligation ID"] = (
            df["Obligation ID"].astype("Int64").astype("string")
        )
        df["Status"] = df["Status"].astype("string").str.strip()

        return df

    anterior = preparar(anterior)
    actual = preparar(actual)

    key_cols = ["Publication ID", "Nombre", "Tier"]
    id_cols = ["Publication ID", "Obligation ID"]

    # evitar agrupaciones o comparaciones ambiguas.
    for nombre_df, df in [("anterior", anterior), ("actual", actual)]:
        if df[["Publication ID"]].isna().any().any():
            raise ValueError(
                f"{nombre_df}: Publication ID contiene nulos."
            )

        if df[key_cols].isna().any().any():
            raise ValueError(
                f"{nombre_df}: las claves de publicación contienen nulos."
            )

    # 2. separar confirmadas y pendientes
    actual_conf = actual.loc[actual["Status"].eq("Confirmado")]
    actual_pend = actual.loc[~actual["Status"].eq("Confirmado")]

    # 3. calcular métricas del inventario actual
    def contar_obligaciones(df, nombre_col):
        return (
            df.groupby(key_cols)["Obligation ID"]
            .nunique()
            .rename(nombre_col)
            .reset_index()
        )

    mapeadas = contar_obligaciones(
        actual_conf, "Obligaciones mapeadas"   # mapedas -> artículos que ya están confirmados
    )
    pendientes = contar_obligaciones(    # pendientes -> artículos que están bajo revisión/análisis
        actual_pend, "Bajo análisis"
    )

    # 4. detectar altas, promociones y bajas
    # se usa una sola referencia de las obligaciones anteriores.
    anterior_ids = anterior[id_cols].drop_duplicates()
    anterior_status = anterior[
        id_cols + ["Status"]
    ].drop_duplicates()

    actual_ids = actual[id_cols].drop_duplicates()

    # altas: confirmadas que no existían anteriormente.
    altas = (
        actual_conf.merge(
            anterior_ids,
            on=id_cols,
            how="left",
            indicator=True
        )
        .loc[lambda df: df["_merge"].eq("left_only")]
        .drop(columns="_merge")
    )

    # promociones: periodo anterior no confirmadas y en el actual confirmadas.
    promovidas = (
        actual_conf.merge(
            anterior_status,
            on=id_cols,
            how="inner",
            suffixes=("_act", "_ant")
        )
        .loc[lambda df: df["Status_ant"].ne("Confirmado")]
    )

    # bajas: obligaciones que estaban antes y ya no existen.
    bajas = (
        anterior.merge(
            actual_ids,
            on=id_cols,
            how="left",
            indicator=True
        )
        .loc[lambda df: df["_merge"].eq("left_only")]
        .drop(columns="_merge")
    )

    # 5. consolidar todas las publicaciones
    status = (
        pd.concat(
            [anterior[key_cols], actual[key_cols]],
            ignore_index=True
        )
        .drop_duplicates()
        .merge(mapeadas, on=key_cols, how="left")
        .merge(pendientes, on=key_cols, how="left")
    )

    metricas = ["Obligaciones mapeadas", "Bajo análisis"]
    status[metricas] = status[metricas].fillna(0).astype(int)

    status["Total de obligaciones"] = status[metricas].sum(axis=1)

    # 6. crear listas únicas de obligaciones por publicación
    def listas_por_publicacion(df):
        if df.empty:
            return {}

        return (
            df.groupby(key_cols)["Obligation ID"]
            .agg(
                lambda s: list(
                    dict.fromkeys(s.dropna().astype(str))
                )
            )
            .to_dict()
        )

    altas_d = listas_por_publicacion(altas)
    promovidas_d = listas_por_publicacion(promovidas)
    bajas_d = listas_por_publicacion(bajas)
    pendientes_d = listas_por_publicacion(actual_pend)

    keys = list(status[key_cols].itertuples(index=False, name=None))

    def obtener_listas(diccionario):
        return [diccionario.get(k, []) for k in keys]

    altas_l = obtener_listas(altas_d)
    promovidas_l = obtener_listas(promovidas_d)
    bajas_l = obtener_listas(bajas_d)
    pendientes_l = obtener_listas(pendientes_d)

    # 7. construir notas, lo cual es el aspecto más relevante de este script
    def combinar_notas(*partes):
        return "; ".join(
            f"{etiqueta}: {', '.join(ids)}"
            for etiqueta, ids in partes
            if ids
        )

    status["Notas"] = [
        combinar_notas(
            ("Alta", list(dict.fromkeys(a + p))),
            ("Baja", b),
            ("Bajo análisis", pend)
        )
        for a, p, b, pend in zip(
            altas_l, promovidas_l, bajas_l, pendientes_l
        )
    ]

    # excepción: si hay Tier 3/4/5 no hay obligaciones/artículos relacionados.
    sin_obligaciones = (
        status["Tier"].isin(["Tier 3", "Tier 4", "Tier 5"])
        & status["Total de obligaciones"].eq(0)
    )
    status.loc[sin_obligaciones, "Notas"] = ""

    # Las altas y bajas de publicaciones tienen prioridad.
    pubs_anteriores = set(anterior["Publication ID"])
    pubs_actuales = set(actual["Publication ID"])

    status.loc[
        status["Publication ID"].isin(
            pubs_anteriores - pubs_actuales
        ),
        "Notas"
    ] = "Baja"

    status.loc[
        status["Publication ID"].isin(
            pubs_actuales - pubs_anteriores
        ),
        "Notas"
    ] = "Alta"

    # 8. resultado final
    columnas_finales = [
        "Publication ID",
        "Nombre",
        "Tier",
        "Total de obligaciones",
        "Obligaciones mapeadas",
        "Bajo análisis",
        "Notas",
    ]

    return status[columnas_finales].reset_index(drop=True)

In [5]:
# probamos la función con los datos de ejemplo: periodo anterior y actual
estatus_inventario(anterior, actual)

,Publication ID,Nombre,Tier,Total de obligaciones,Obligaciones mapeadas,Bajo análisis,Notas
0,3345,Ley Renta,Tier 1,2,2,0,Alta: 2488
1,3467,Ley IVA,Tier 1,3,2,1,Baja: 2105; Bajo análisis: 2990
2,3876,Reglamento XYZ,Tier 2,3,2,1,Bajo análisis: 2991
3,3965,Circular Y,Tier 2,3,3,0,"Alta: 2048, 2638"
4,3321,Codigo fiscal,Tier 3,0,0,0,Baja
5,3999,Miscelanea fiscal,Tier 3,0,0,0,Alta


Lo sé, probablemente hasta aquí no hayas digerido completamente cual fue la intención de todo el código pero lo trataré de explicar con imagenes, así se ve el estatus inicial de los datos:

<img src="images/estatus.png" alt="estatus" title="estatus inicial tanto para el periodo anterior como el actual" width="900">

El resultado final es una **transformación** radical,  puesto que condensa el número total de artículos confirmados en el inventario regulatorio, el número total de artículos bajo análisis, por último en la columna Notas resume lo que pasó en el periodo actual respecto al anterior.

En resumen, el código toma listas de estatus, internamente con reglas de negocio y con lógica transforma dichos estatus en la presentación final, donde en la columna **Notas** presenta de forma condensada si hay:
* Altas de artículos en el periodo, ejemplo Alta: 2488
* Bajas de artículos en el periodo, ejemplo Baja: 2105
* Artículos que están en revisión de aplicabilidad, ejemplo Bajo análisis: 2990
* Normatividad que ha sido dado de baja en el periodo, ejemplo Baja del Código fiscal
* Normatividad dada de alta en el periodo, ejemplo Alta de la Miscelanea fiscal

Para terminar, veo varios puntos positivos en invertir tiempo para automatizar una tarea así:

1. Partiendo de un histórico y guardándolo para que sirva como punto de inicio, no debes preocuparte por la versión de Excel o preocuparte por ¿quien tiene la última versión? **Hay consistencia y documentación del trabajo**
2. Condensando toda la lógica resulta trivial ejecutar un notebook en 2-3 minutos, copiar la tabla resultante y llevarla a esa presentación para Comité o para alta dirección, en lugar de un trabajo manual cada mes con la preocupación de las múltiples versiones Excel que puedan existir, lógica utilizada en Excel, la cual no es en absoluto transparente y ni hablar de las formulas que no son tan descriptivas. **Con python hay reproducibilidad**
3. En mi experiencia, implementar esto me ahorró horas de trabajo, presenté resultados de una forma más eficiente y sobre todo, teniendo datos actualizados la generación de la tabla final, vía la función creada se puede hacer _on demand_, no necesita mayor preparación, asignación de recursos, etc. Tan simple como que llega un regulador, auditor, etc. y pide el inventario regulatorio actualizado y lo tienes a un click de distancia. **Con una solución de este tipo te vuelves estratégico**